# Trajectory features

Summing up the trajectory points of each interval (altitude, speeds) to use them as features. Tested on one flight first, then on all of them.

In [ ]:
from prc_fuel.load import q

In [ ]:
# trajectory summary per interval, one flight
q("""
SELECT
  f.idx,
  COUNT(ft.flight_id) AS n_points,
  AVG(ft.altitude) AS avg_altitude,
  MIN_BY(ft.altitude, IF(ft.altitude IS NULL, NULL, ft.`timestamp`)) AS alt_start,
  MAX_BY(ft.altitude, IF(ft.altitude IS NULL, NULL, ft.`timestamp`)) AS alt_end,
  AVG(ft.groundspeed) AS avg_groundspeed,
  AVG(ft.vertical_rate) AS avg_vertical_rate,
  AVG(ft.mach) AS avg_mach
FROM `prc-data-challenge-2025.prc2025.fuel_train` AS f
LEFT JOIN `prc-data-challenge-2025.prc2025.flights_train` AS ft
  ON ft.flight_id = f.flight_id
  AND ft.`timestamp` BETWEEN f.start AND f.`end`
WHERE f.flight_id = 'prc770822360'
GROUP BY f.idx
ORDER BY f.idx
""")

Intervals 0 to 3 are cruise, ~36000 ft and vertical rate close to 0. Interval 4 is the descent, 36000 -> 15225 ft. The mean altitude alone (29500) didn't show how much it went down, hence alt_start / alt_end.
Units: ft, kt, ft/min.

In [ ]:
# points sitting exactly on the interval bounds
q("""
SELECT
  f.idx,
  ft.`timestamp` = f.start AS at_start,
  ft.altitude,
  ft.groundspeed,
  ft.mach,
  ft.source
FROM `prc-data-challenge-2025.prc2025.fuel_train` AS f
JOIN `prc-data-challenge-2025.prc2025.flights_train` AS ft
  ON ft.flight_id = f.flight_id
  AND (ft.`timestamp` = f.start OR ft.`timestamp` = f.`end`)
WHERE f.flight_id = 'prc770822360'
ORDER BY f.idx, at_start DESC
""")

On this flight every interval has a point exactly on start and on end (not checked on other flights yet). They come from acars, have a mach but no altitude or groundspeed. So a plain MIN_BY(altitude, timestamp) picks them and returns null, that's why the IF skips points without altitude.
Bonus: mach is there, so we get some airspeed after all.

In [ ]:
# same summary on all flights, only the row count (~4.7 GB read)
q("""
SELECT COUNT(*) AS n_rows
FROM (
  SELECT
    f.flight_id,
    f.idx,
    COUNT(ft.flight_id) AS n_points,
    AVG(ft.altitude) AS avg_altitude,
    MIN_BY(ft.altitude, IF(ft.altitude IS NULL, NULL, ft.`timestamp`)) AS alt_start,
    MAX_BY(ft.altitude, IF(ft.altitude IS NULL, NULL, ft.`timestamp`)) AS alt_end,
    AVG(ft.groundspeed) AS avg_groundspeed,
    AVG(ft.vertical_rate) AS avg_vertical_rate,
    AVG(ft.mach) AS avg_mach
  FROM `prc-data-challenge-2025.prc2025.fuel_train` AS f
  LEFT JOIN `prc-data-challenge-2025.prc2025.flights_train` AS ft
    ON ft.flight_id = f.flight_id
    AND ft.`timestamp` BETWEEN f.start AND f.`end`
  GROUP BY f.flight_id, f.idx
)
""")

131530 rows, same as fuel_train, so no interval lost or duplicated by the join. Took ~5 s in BigQuery Studio for 4.7 GB read.